![RuG Logo](https://www.rug.nl/about-ug/practical-matters/huisstijl/logobank-new/logo-faculteiten/eng-logo/horizontal/rood/png/rugr_fse_logoen_rood_rgb.png)

---

# **Lab 01 | Data Privacy Course | Fall 2026**




```
Course Instructor: Prof. Dr. Fatih Turkmen
TAs: Pablo Sorrentino, Ali Satvaty
```
---

# Membership Inference Attacks

**Data Privacy / Machine Learning Security and Privacy**

This lab is a practical continuation of the lecture on Membership Inference Attacks (MIAs).

A membership inference attacker asks:

> **Was a particular sample used to train the target model?**

We study two scenarios:

1. **Scenario 1 — Simple black-box attacks**
   - correctness / label-only baseline;
   - confidence- and loss-based membership scores;
   - compare a less-regularized and a more-regularized target model.

2. **Scenario 2 — Shadow-model attack**
   - train a shadow model on auxiliary data;
   - create labelled `member` / `non-member` examples;
   - train attack classifiers;
   - attack a separate target model.

The design is intentionally lightweight: it uses only `scikit-learn`, `numpy`, `pandas`, and `matplotlib`, runs on CPU, requires no external downloads, and uses fixed random seeds.

---

### Where this lab design comes from

The structure follows ideas used in:

- Shokri et al., **Membership Inference Attacks Against Machine Learning Models**, IEEE S&P 2017.
- Yeom et al., **Privacy Risk in Machine Learning: Analyzing the Connection to Overfitting**, CSF 2018.
- MIT Introduction to Data-Centric AI, **Data Privacy and Security / Membership Inference lab**.
- Adversarial Robustness Toolbox (ART) public MIA tutorials:
  - rule-based black-box MIA;
  - learned black-box MIA;
  - shadow-model MIA;
  - label-only decision-boundary MIA.

We reproduce the ideas transparently rather than depending on ART during the lab.

## Learning objectives

By the end of this lab, you should be able to:

- identify the **target model**, **members**, and **non-members**;
- state the attacker's **goal, knowledge, and observable outputs**;
- explain why model behaviour can leak training-set membership;
- interpret **ROC AUC** as a measure of membership leakage;
- explain what a **shadow model** is and why an attacker might use one;
- distinguish a simple rule-based attack from a learned MIA;
- discuss the relation between generalization behaviour and membership leakage.

### Workflow

**READ → RUN → OBSERVE → INTERPRET → ANSWER**

You do **not** need to understand every implementation detail. Focus on the attacker's perspective and the meaning of the outputs.

# Threat model

Before discussing an attack, specify what the adversary can and cannot do.

### Attacker CAN

- query the target model with chosen samples;
- observe the model's predicted class probabilities;
- know the true class label of the sample being tested;
- obtain auxiliary data from approximately the same distribution;
- train their own models.

### Attacker CANNOT

- inspect the target model's private training set;
- directly know whether a queried sample was used for training;
- modify the target model in this lab.

This is primarily a **black-box membership inference** setting.

> The key question is: **can model outputs reveal whether a sample was in training?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Dataset

We use scikit-learn's built-in **Digits** dataset.

Each sample is an 8×8 grayscale image of a handwritten digit from 0 to 9.

Why use this dataset for a lab?

- no download is needed;
- it trains in seconds on CPU;
- all students work with exactly the same data;
- we can focus on privacy rather than infrastructure.

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target

print("Samples:", len(X))
print("Features per sample:", X.shape[1])
print("Classes:", len(np.unique(y)))

fig, axes = plt.subplots(1, 5, figsize=(9, 2))
for ax, idx in zip(axes, range(5)):
    ax.imshow(digits.images[idx], cmap="gray")
    ax.set_title(f"Label: {y[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Scenario 1: Simple black-box membership inference

We begin with attacks that do **not** train a separate attack classifier.

We compare two Random Forest target models:

- **Less regularized** (higher capacity);
- **More regularized** (constrained depth and larger leaves).

We will explicitly measure both:

1. how differently the target behaves on train vs. test data;
2. how much membership information an attacker can extract.

This avoids assuming that one hyperparameter automatically means "more overfitting".

In [ ]:
# Use a modest training set so that membership leakage is visible in a short lab.
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X,
    y,
    train_size=500,
    test_size=800,
    stratify=y,
    random_state=RANDOM_STATE,
)

less_regularized = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

more_regularized = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=8,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

less_regularized.fit(X_target_train, y_target_train)
more_regularized.fit(X_target_train, y_target_train)

print("Target train size:", len(X_target_train))
print("Target test size:", len(X_target_test))

In [ ]:
def summarize_target(model):
    train_prob = model.predict_proba(X_target_train)
    test_prob = model.predict_proba(X_target_test)

    return {
        "Train accuracy": accuracy_score(y_target_train, model.predict(X_target_train)),
        "Test accuracy": accuracy_score(y_target_test, model.predict(X_target_test)),
        "Accuracy gap": (
            accuracy_score(y_target_train, model.predict(X_target_train))
            - accuracy_score(y_target_test, model.predict(X_target_test))
        ),
        "Train log-loss": log_loss(y_target_train, train_prob, labels=model.classes_),
        "Test log-loss": log_loss(y_target_test, test_prob, labels=model.classes_),
        "Loss gap": (
            log_loss(y_target_test, test_prob, labels=model.classes_)
            - log_loss(y_target_train, train_prob, labels=model.classes_)
        ),
    }

target_summary = pd.DataFrame({
    "Less regularized": summarize_target(less_regularized),
    "More regularized": summarize_target(more_regularized),
}).T

target_summary.round(3)

## 1A: Rule-based / correctness attack

A very simple baseline is:

> **If the model predicts the sample correctly, guess MEMBER. Otherwise, guess NON-MEMBER.**

Why might this ever work?

If the target has higher accuracy on its training set than on unseen data, correctness itself contains a small membership signal.

This is a deliberately weak baseline — but it is useful because it shows how little information an attacker may need.

In [ ]:
def balanced_membership_evaluation(model):
    # Use the same number of members and non-members.
    n = min(len(X_target_train), len(X_target_test))
    local_rng = np.random.default_rng(RANDOM_STATE)

    member_idx = local_rng.choice(len(X_target_train), n, replace=False)
    nonmember_idx = local_rng.choice(len(X_target_test), n, replace=False)

    X_eval = np.vstack([
        X_target_train[member_idx],
        X_target_test[nonmember_idx],
    ])
    y_eval = np.concatenate([
        y_target_train[member_idx],
        y_target_test[nonmember_idx],
    ])
    membership = np.concatenate([
        np.ones(n, dtype=int),
        np.zeros(n, dtype=int),
    ])
    return X_eval, y_eval, membership

rule_results = {}

for name, model in [
    ("Less regularized", less_regularized),
    ("More regularized", more_regularized),
]:
    X_eval, y_eval, membership_true = balanced_membership_evaluation(model)

    # Rule: correct prediction -> member
    membership_pred = (model.predict(X_eval) == y_eval).astype(int)

    rule_results[name] = {
        "Rule-based MIA accuracy": accuracy_score(membership_true, membership_pred)
    }

pd.DataFrame(rule_results).T.round(3)

## 1B: Confidence- and loss-based attacks

Correctness throws away information.

If the model exposes probabilities, the attacker can inspect **how confident** it is.

For each labelled sample \((x,y)\), define:

- **confidence score:** probability assigned to the true class \(y\);
- **loss score:** negative per-sample log-loss, so that a larger score means "more member-like".

Then evaluate how well each score separates:

- members;
- non-members.

We use **ROC AUC**:

- **0.5** ≈ random ranking;
- **1.0** = perfect separation;
- higher AUC = stronger membership leakage.

In [ ]:
def true_class_probability(model, X_data, y_data):
    prob = model.predict_proba(X_data)
    return prob[np.arange(len(y_data)), y_data]

def score_based_mia(model):
    member_conf = true_class_probability(model, X_target_train, y_target_train)
    nonmember_conf = true_class_probability(model, X_target_test, y_target_test)

    membership = np.concatenate([
        np.ones(len(member_conf)),
        np.zeros(len(nonmember_conf)),
    ])

    confidence_scores = np.concatenate([member_conf, nonmember_conf])

    # Per-sample cross-entropy loss is -log(p_true).
    # Negating it makes "larger = more member-like", consistent with confidence.
    member_loss_score = np.log(np.clip(member_conf, 1e-12, 1.0))
    nonmember_loss_score = np.log(np.clip(nonmember_conf, 1e-12, 1.0))
    loss_scores = np.concatenate([member_loss_score, nonmember_loss_score])

    return {
        "Confidence AUC": roc_auc_score(membership, confidence_scores),
        "Loss-based AUC": roc_auc_score(membership, loss_scores),
        "Mean member confidence": member_conf.mean(),
        "Mean non-member confidence": nonmember_conf.mean(),
        "member_conf": member_conf,
        "nonmember_conf": nonmember_conf,
    }

score_results = {}

for name, model in [
    ("Less regularized", less_regularized),
    ("More regularized", more_regularized),
]:
    out = score_based_mia(model)
    score_results[name] = {
        k: v for k, v in out.items()
        if k not in ("member_conf", "nonmember_conf")
    }

pd.DataFrame(score_results).T.round(3)

In [ ]:
# Visualize the membership signal for the less-regularized model.
out = score_based_mia(less_regularized)

plt.figure(figsize=(7, 4))
plt.hist(out["member_conf"], bins=20, alpha=0.6, density=True, label="Members")
plt.hist(out["nonmember_conf"], bins=20, alpha=0.6, density=True, label="Non-members")
plt.xlabel("Probability assigned to the true class")
plt.ylabel("Density")
plt.title("Target confidence: members vs. non-members")
plt.legend()
plt.show()

In [ ]:
# Here it is possible to explore the members!

GROUP = "non-member"      # Try: "member" or "non-member"
SAMPLE_INDEX = 0      # Try: 0, 1, 2, 3, ...

if GROUP == "member":
    sample = X_target_train[SAMPLE_INDEX]
    label = y_target_train[SAMPLE_INDEX]
    true_membership = "MEMBER"
else:
    sample = X_target_test[SAMPLE_INDEX]
    label = y_target_test[SAMPLE_INDEX]
    true_membership = "NON-MEMBER"

probabilities = less_regularized.predict_proba([sample])[0]

prediction = less_regularized.predict([sample])[0]
confidence = probabilities[label]

plt.figure(figsize=(3, 3))
plt.imshow(sample.reshape(8, 8), cmap="gray")
plt.axis("off")
plt.title(
    f"True digit: {label}\n"
    f"Prediction: {prediction}\n"
    f"Confidence: {confidence:.3f}\n"
    f"True membership: {true_membership}"
)
plt.show()

In [ ]:
# Here it is possible to check the trade-off between threshold, precision and recall!

THRESHOLD = 0.90   # Try: 0.70, 0.80, 0.90, 0.95, 0.99

member_scores = true_class_probability(
    less_regularized,
    X_target_train,
    y_target_train
)

nonmember_scores = true_class_probability(
    less_regularized,
    X_target_test,
    y_target_test
)

scores = np.concatenate([
    member_scores,
    nonmember_scores
])

true_membership = np.concatenate([
    np.ones(len(member_scores)),
    np.zeros(len(nonmember_scores))
])

predicted_membership = (scores >= THRESHOLD).astype(int)

accuracy = accuracy_score(
    true_membership,
    predicted_membership
)

precision = precision_score(
    true_membership,
    predicted_membership,
    zero_division=0
)

recall = recall_score(
    true_membership,
    predicted_membership,
    zero_division=0
)

print("Threshold:", THRESHOLD)
print("Attack accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))

In [ ]:
results = pd.Series({
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall
})

plt.figure(figsize=(6, 4))
results.plot(kind="bar")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title(f"MIA performance at threshold = {THRESHOLD}")
plt.xticks(rotation=0)
plt.show()

### Training size vs Privacy Leakage

It is possible to change `TRAIN_SIZES` and observe how the membership inference AUC changes.

In [ ]:
TRAIN_SIZES = [100, 200, 300, 500, 700, 900]

experiment_results = []

for train_size in TRAIN_SIZES:

    X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
        X,
        y,
        train_size=train_size,
        test_size=800,
        stratify=y,
        random_state=RANDOM_STATE
    )

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1,
        max_features=None,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_exp, y_train_exp)

    train_acc = accuracy_score(
        y_train_exp,
        model.predict(X_train_exp)
    )

    test_acc = accuracy_score(
        y_test_exp,
        model.predict(X_test_exp)
    )

    member_scores = true_class_probability(
        model,
        X_train_exp,
        y_train_exp
    )

    nonmember_scores = true_class_probability(
        model,
        X_test_exp,
        y_test_exp
    )

    membership_labels = np.concatenate([
        np.ones(len(member_scores)),
        np.zeros(len(nonmember_scores))
    ])

    membership_scores = np.concatenate([
        member_scores,
        nonmember_scores
    ])

    mia_auc = roc_auc_score(
        membership_labels,
        membership_scores
    )

    experiment_results.append({
        "Training size": train_size,
        "Train accuracy": train_acc,
        "Test accuracy": test_acc,
        "MIA AUC": mia_auc
    })

privacy_df = pd.DataFrame(experiment_results)

display(privacy_df.round(3))

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    privacy_df["Training size"],
    privacy_df["MIA AUC"],
    marker="o"
)

plt.axhline(
    0.5,
    linestyle="--",
    label="Random guessing"
)

plt.xlabel("Number of training samples")
plt.ylabel("Membership Inference AUC")
plt.title("Training Set Size vs Privacy Leakage")
plt.legend()

plt.show()

### Questions from Scenario 1

**Q1.** Is the rule-based attack substantially better than random guessing? Why or why not?

**Q2.** Which target model leaks more membership information according to ROC AUC?

**Q3.** Compare mean confidence for members and non-members. What do you observe?

**Q4.** Does high confidence prove membership? Explain.

**Q5.** Compare the train/test accuracy and log-loss gaps with the MIA AUCs. What does this experiment suggest about generalization behaviour and membership leakage?

**Q6.** Why is ROC AUC more informative here than reporting only one hard threshold and one accuracy value?

# Scenario 2: Shadow-model membership inference

Now we reproduce the central idea from the lecture.

The attacker does not know the true membership labels of the target model's private data.

So how can the attacker train a `member / non-member` classifier?

### Shadow training

The attacker obtains auxiliary data from a similar population and trains a **shadow model**.

For the shadow model:

- samples used to train it are known **members**;
- held-out samples are known **non-members**.

The attacker queries the shadow model on both groups and obtains a labelled meta-dataset:

`model outputs → MEMBER / NON-MEMBER`

Then an attack classifier learns the difference.

Finally:

`target model output → attack classifier → membership prediction`

## Important separation of data

We will create two disjoint pools:

- **TARGET pool:** used only to train/evaluate the victim model;
- **SHADOW pool:** auxiliary data available to the attacker.

The attacker never uses the target model's private training membership labels to train the attack classifier.

In [ ]:
X_target_pool, X_shadow_pool, y_target_pool, y_shadow_pool = train_test_split(
    X,
    y,
    test_size=0.50,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_t_train, X_t_test, y_t_train, y_t_test = train_test_split(
    X_target_pool,
    y_target_pool,
    test_size=0.45,
    stratify=y_target_pool,
    random_state=1,
)

X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_shadow_pool,
    y_shadow_pool,
    test_size=0.45,
    stratify=y_shadow_pool,
    random_state=2,
)

print("TARGET model:")
print("  private members:", len(X_t_train))
print("  held-out non-members:", len(X_t_test))

print("\nSHADOW model:")
print("  attacker-known members:", len(X_s_train))
print("  attacker-known non-members:", len(X_s_test))

In [ ]:
rf_parameters = dict(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features=None,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

target_model = RandomForestClassifier(**rf_parameters)
shadow_model = RandomForestClassifier(**rf_parameters)

target_model.fit(X_t_train, y_t_train)
shadow_model.fit(X_s_train, y_s_train)

print(
    "Target train/test accuracy:",
    round(accuracy_score(y_t_train, target_model.predict(X_t_train)), 3),
    "/",
    round(accuracy_score(y_t_test, target_model.predict(X_t_test)), 3),
)

print(
    "Shadow train/test accuracy:",
    round(accuracy_score(y_s_train, shadow_model.predict(X_s_train)), 3),
    "/",
    round(accuracy_score(y_s_test, shadow_model.predict(X_s_test)), 3),
)

## Build the attack training data

ART's public shadow-model tutorial generates a meta-dataset from shadow-model predictions.

Here we implement the same idea directly.

Following the formulation shown in the course lecture, we train **one binary attack classifier per output class**.

Each attack classifier receives the shadow model's probability vector and predicts:

- `1` = member
- `0` = non-member

In [ ]:
shadow_member_outputs = shadow_model.predict_proba(X_s_train)
shadow_nonmember_outputs = shadow_model.predict_proba(X_s_test)

attack_models = {}

for class_id in range(10):
    member_outputs_c = shadow_member_outputs[y_s_train == class_id]
    nonmember_outputs_c = shadow_nonmember_outputs[y_s_test == class_id]

    # Balance member and non-member examples for this class.
    n = min(len(member_outputs_c), len(nonmember_outputs_c))
    local_rng = np.random.default_rng(100 + class_id)

    member_idx = local_rng.choice(len(member_outputs_c), n, replace=False)
    nonmember_idx = local_rng.choice(len(nonmember_outputs_c), n, replace=False)

    X_attack = np.vstack([
        member_outputs_c[member_idx],
        nonmember_outputs_c[nonmember_idx],
    ])
    y_attack = np.concatenate([
        np.ones(n),
        np.zeros(n),
    ])

    attack_model = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    attack_model.fit(X_attack, y_attack)
    attack_models[class_id] = attack_model

print("Attack classifiers trained:", len(attack_models))

## Attack the target model

Now query the separate **target model**.

For evaluation purposes *we* know which samples are members and non-members, but the attack classifier only receives the target model's probability outputs.

We use a balanced evaluation set so that attack accuracy is easy to interpret.

In [ ]:
target_member_outputs = target_model.predict_proba(X_t_train)
target_nonmember_outputs = target_model.predict_proba(X_t_test)

n_eval = min(len(target_member_outputs), len(target_nonmember_outputs))
eval_rng = np.random.default_rng(RANDOM_STATE)

member_idx = eval_rng.choice(len(target_member_outputs), n_eval, replace=False)
nonmember_idx = eval_rng.choice(len(target_nonmember_outputs), n_eval, replace=False)

target_outputs_eval = np.vstack([
    target_member_outputs[member_idx],
    target_nonmember_outputs[nonmember_idx],
])

target_classes_eval = np.concatenate([
    y_t_train[member_idx],
    y_t_test[nonmember_idx],
])

membership_true = np.concatenate([
    np.ones(n_eval),
    np.zeros(n_eval),
])

membership_score = np.empty(len(membership_true))

for class_id in range(10):
    mask = target_classes_eval == class_id
    membership_score[mask] = attack_models[class_id].predict_proba(
        target_outputs_eval[mask]
    )[:, 1]

membership_pred = (membership_score >= 0.5).astype(int)

shadow_metrics = {
    "ROC AUC": roc_auc_score(membership_true, membership_score),
    "Accuracy": accuracy_score(membership_true, membership_pred),
    "Precision": precision_score(membership_true, membership_pred),
    "Recall": recall_score(membership_true, membership_pred),
    "F1": f1_score(membership_true, membership_pred),
}

pd.Series(shadow_metrics).round(3)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(
    membership_score[membership_true == 1],
    bins=20,
    alpha=0.6,
    density=True,
    label="True members",
)
plt.hist(
    membership_score[membership_true == 0],
    bins=20,
    alpha=0.6,
    density=True,
    label="True non-members",
)
plt.xlabel("Attack model's predicted probability of membership")
plt.ylabel("Density")
plt.title("Shadow-model MIA against the target model")
plt.legend()
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(
    membership_true,
    membership_score
)

auc = roc_auc_score(
    membership_true,
    membership_score
)

plt.figure(figsize=(6, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Shadow MIA (AUC = {auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random guessing"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Membership Inference ROC Curve")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

### Optional advanced metric — performance at low false-positive rate

For privacy attacks, average performance is not the whole story.

A realistic attacker may care about making **few false accusations** of membership. Some modern MIA evaluations therefore inspect the true-positive rate (TPR) at a small allowed false-positive rate (FPR).

The ART tutorials include this type of ROC-oriented evaluation for learned black-box MIAs.

This cell is optional for the main lab.

In [ ]:
fpr, tpr, thresholds = roc_curve(membership_true, membership_score)

def best_tpr_at_fpr(max_fpr):
    valid = np.where(fpr <= max_fpr)[0]
    return tpr[valid].max() if len(valid) else 0.0

print("TPR at FPR <= 1% :", round(best_tpr_at_fpr(0.01), 3))
print("TPR at FPR <= 5% :", round(best_tpr_at_fpr(0.05), 3))

### Questions — Scenario 2

**Q7.** Why does the attacker train a shadow model?

**Q8.** Which samples become `member` and `non-member` examples for training the attack classifier?

**Q9.** What does each attack classifier receive as input?

**Q10.** Is the shadow-model attack perfect? Use ROC AUC and at least one other metric.

**Q11.** Why do we keep the TARGET and SHADOW pools separate?

**Q12.** Why might TPR at a low FPR be useful in addition to overall ROC AUC?

**Q13.** Give one real-world example where learning that a person was a member of a training set could itself reveal sensitive information.

# Optional: What if probabilities are hidden?

The main exercises assume that the API exposes class probabilities.

But hiding probabilities does **not automatically eliminate membership leakage**.

ART also provides a **label-only decision-boundary MIA**. Its intuition is:

> Training samples can sometimes lie farther from the decision boundary than non-members.

An attacker can perturb a sample and estimate how much perturbation is required to change the model's predicted label.

This is a stronger and more computationally expensive attack, so we do not implement it in the core lab.

### Discussion question

If an API returned only the top-1 class label instead of probabilities, which of today's attacks would stop working directly, and which membership signal might still remain?

# Optional Example: Membership Inference on Tabular Medical Data

In [ ]:
from sklearn.datasets import load_breast_cancer

medical = load_breast_cancer()

X_med = medical.data
y_med = medical.target

print("Samples:", X_med.shape[0])
print("Features:", X_med.shape[1])
print("Classes:", medical.target_names)

In [ ]:
X_med_train, X_med_test, y_med_train, y_med_test = train_test_split(
    X_med,
    y_med,
    train_size=120,
    test_size=300,
    stratify=y_med,
    random_state=RANDOM_STATE
)

medical_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features=None,
    random_state=RANDOM_STATE
)

medical_model.fit(
    X_med_train,
    y_med_train
)

## MIA

In [ ]:
member_scores = true_class_probability(
    medical_model,
    X_med_train,
    y_med_train
)

nonmember_scores = true_class_probability(
    medical_model,
    X_med_test,
    y_med_test
)

membership_labels = np.concatenate([
    np.ones(len(member_scores)),
    np.zeros(len(nonmember_scores))
])

membership_scores = np.concatenate([
    member_scores,
    nonmember_scores
])

medical_auc = roc_auc_score(
    membership_labels,
    membership_scores
)

print("Medical-data MIA AUC:", round(medical_auc, 3))

In [ ]:
plt.figure(figsize=(7, 4))

plt.hist(
    member_scores,
    bins=20,
    alpha=0.6,
    density=True,
    label="Members"
)

plt.hist(
    nonmember_scores,
    bins=20,
    alpha=0.6,
    density=True,
    label="Non-members"
)

plt.xlabel("True-class confidence")
plt.ylabel("Density")
plt.title(
    f"Medical tabular data — MIA AUC = {medical_auc:.3f}"
)

plt.legend()
plt.show()

# Final comparison

| Attack | Target access used | Extra attacker training? | Main idea |
|---|---|---:|---|
| Correctness baseline | predicted label + true label | No | correct ⇒ guess member |
| Confidence/loss MIA | class probabilities + true label | No | members may receive different scores |
| Shadow-model MIA | class probabilities + auxiliary data | Yes | learn member/non-member output patterns |
| Label-only decision-boundary MIA | predicted labels | Yes/calibration | estimate distance to decision boundary |

### Key takeaways

1. **Membership is a prediction problem:** `MEMBER` vs. `NON-MEMBER`.
2. A target model can leak membership through its observable behaviour.
3. Correctness alone may leak a little; probabilities can leak more.
4. Shadow models let an attacker manufacture labelled membership examples without seeing the target's private training set.
5. Better predictive accuracy does **not** automatically imply privacy.
6. Generalization behaviour and privacy leakage are related, but should be **measured**, not assumed.
7. The attack does not need to be perfect to create a privacy risk.

# References

### Research papers

- Reza Shokri, Marco Stronati, Congzheng Song, Vitaly Shmatikov. **Membership Inference Attacks Against Machine Learning Models.** IEEE Symposium on Security and Privacy, 2017.
- Samuel Yeom, Irene Giacomelli, Matt Fredrikson, Somesh Jha. **Privacy Risk in Machine Learning: Analyzing the Connection to Overfitting.** IEEE Computer Security Foundations Symposium, 2018.

### Teaching / implementation references

- **MIT Introduction to Data-Centric AI — Data Privacy and Security.** The public lab asks students to attack a trained black-box model and determine training-set membership.

- **Stanford CS329T — Trustworthy Machine Learning.** Public course material treats membership inference as part of ML privacy and contrasts black-box / white-box strategies.

The implementation in this notebook is intentionally written from scratch with scikit-learn so that every important step of the attack remains visible.